# STFPM 새 구조 — 2단계(v3.2) 정상 사진 학습, 원본 / 질감 마스크 (시간 블록 5-fold)

**왜 하나**
- STFPM(정상 사진만 학습하는 이상 탐지)을 지도학습 모델들과 **같은 선별·같은 분할·같은 평가 사진**에서 다시 잰다.
- 팀원이 구조를 바꿨다고 한 버전(layer2+layer3, 20 epoch)을 우리 쪽에서 먼저 재현해 둔다. 팀원 코드가 오면 설정을 맞춰 보고 그때 공유한다.
- 지난번 결과 [확인]: 평균 집계(mean)에 마스킹을 하면 부분 표면 사진의 점수가 회색 영역에 묽어져서 재현율이 77.6% → 28.8%로 떨어졌다.
  그래서 마스크 조건은 **상위 5% 평균(topk_mean)** 으로 점수를 낸다. 회색 영역이 넓어도 이상한 부분 몇 %만 보고 판정한다.

| 조건 | 학습 (정상만) | 입력 | 점수 집계 |
|---|---|---|---|
| ① s2_raw_mean | 각 fold 학습 구간 중 2단계 v3.2 통과 정상 | 고정 크롭 1000×1000 그대로 | mean (팀원 기본값) |
| ② s2_texture_topk | 같음 | 같은 크롭, 질감 마스크 밖 회색 | topk_mean (상위 5%) |

- 두 조건 모두 layer2+layer3, 20 epoch, 기준 = 학습 정상 점수의 95% 지점, 가장자리 5% 제외. 나머지는 팀원 v2 코드(10/1) 그대로다.
- 평가는 각 fold 검증 구간 **4,000장 전부** (선별 밖 사진 포함). 순위 비교는 공통 사진 3,039장에서만 한다 (DINOv2 격자·ConvNeXt와 같은 사진).
- ①→② 사이에 마스크와 집계 방식이 **둘 다** 바뀐다. 차이를 마스크 하나의 효과로 말할 수 없다. (마스크 + mean은 지난번에 이미 망가지는 걸 확인했다.)

**Drive `산공종설/`에 필요한 파일**
| 파일 | 상태 |
|---|---|
| underfill_originals_4000.zip | 이미 있음 |
| stfpm_v2_timeblock_mask_bundle_20261006.zip (팀원 STFPM v2 사본 + 실행 래퍼) | 이미 있음 |
| dinov2_grid_bundle_20261013.zip (2단계 v3.2 목록, folds.csv) | 이미 있음 |
| convnext_three_preproc_patch_20261005.zip (질감 마스크) | 이미 있음 |
| **stfpm_stage2v32_bundle_20261013.zip** (입력 준비 + 설정 2개) | **새로 올림** |

**시간 [추정]:** 설치·입력 준비 15분 안팎, 조건마다 40분~1시간 (지난번 layer3·15 epoch 두 조건이 L4에서 약 1시간 20분). 끊기면 다시 실행해도 끝난 fold는 건너뛴다.

## 1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "GPU 런타임이 아님: 런타임 > 런타임 유형 변경 > GPU"

## 2. Drive 연결 + 파일 확인

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os
DRIVE_DIR = "/content/drive/MyDrive/산공종설"
RESULTS = f"{DRIVE_DIR}/results/stfpm_stage2v32_20261013"
os.makedirs(RESULTS, exist_ok=True)
FILES = ("underfill_originals_4000.zip", "stfpm_v2_timeblock_mask_bundle_20261006.zip", "dinov2_grid_bundle_20261013.zip",
         "convnext_three_preproc_patch_20261005.zip", "stfpm_stage2v32_bundle_20261013.zip")
for f in FILES:
    print("OK  " if os.path.exists(f"{DRIVE_DIR}/{f}") else "없음", f)

## 3. 압축 해제
마지막 줄에 원본 `4000`이 나와야 한다.

In [ ]:
import shutil
for f in FILES:
    shutil.copy2(f"{DRIVE_DIR}/{f}", "/content/")
!unzip -q -o /content/underfill_originals_4000.zip -d /content/orig
!rm -rf /content/stfpm_v2 /content/grid /content/texpatch /content/s2
!unzip -q -o /content/stfpm_v2_timeblock_mask_bundle_20261006.zip -d /content/stfpm_v2
!unzip -q -o /content/dinov2_grid_bundle_20261013.zip -d /content/grid
!unzip -q -o /content/convnext_three_preproc_patch_20261005.zip -d /content/texpatch
!unzip -q -o /content/stfpm_stage2v32_bundle_20261013.zip -d /content/s2
!find /content/orig -name "*.jpg" | wc -l

## 4. 패키지 설치
지난번 STFPM 실행과 같은 버전(anomalib 2.6.2)이다. 설치 경고는 무시해도 된다.

In [ ]:
!pip install -q "anomalib==2.6.2" timm
import importlib.metadata as md
print({p: md.version(p) for p in ("anomalib", "torch", "lightning")})
import timm
timm.create_model("resnet18", pretrained=True)   # 팀원 train.py는 HF_HUB_OFFLINE=1로 돌아서, 사전학습 가중치를 미리 받아 둔다
print("resnet18 가중치 준비 완료")

## 5. 입력 준비 (원본 PNG + 질감 마스크 PNG, 4,000장씩, 10분 안팎)
fold마다 `학습 정상` 수가 1272 / 1140 / 1070 / 1202 / 1206, `평가` 800이 나와야 한다 (ConvNeXt 2단계 조건의 학습 정상 수와 같다).

In [ ]:
!python /content/s2/prepare_stfpm_s2.py --self-test
!python /content/s2/prepare_stfpm_s2.py --grid /content/grid --texture /content/texpatch/texture --orig /content/orig --out /content/stfpm_inputs

## 6. 학습 + 점수: ① 원본 × mean
fold별 학습 로그는 Drive `results/.../s2_raw_mean/foldK/train_log.txt`에 저장되고, 화면에는 기준값과 fold 결과만 나온다.
`--max-epochs`는 설정 파일의 20을 그대로 쓴다 (팀원 코드 상한도 20이라 제한 해제(--relax)가 필요 없다).

In [ ]:
COMMON = ("--project /content/stfpm_v2/STFPM_Defect_Detection --orig-root /content/stfpm_inputs --no-roi-crop "
          "--split-csv /content/stfpm_inputs/split_s2v32.csv --work /content/stfpm_work --train-args '--percentile 0.95'")
!python /content/stfpm_v2/stfpm_timeblock.py {COMMON} --path-column raw_crop --model-config /content/s2/configs/stfpm_l23_mean.yaml --out "{RESULTS}/s2_raw_mean"

## 7. 학습 + 점수: ② 질감 마스크 × topk_mean

In [ ]:
!python /content/stfpm_v2/stfpm_timeblock.py {COMMON} --path-column texture_crop --model-config /content/s2/configs/stfpm_l23_topk_mean.yaml --out "{RESULTS}/s2_texture_topk"

## 8. 결과 (DINOv2 격자·ConvNeXt와 같은 사진 묶음)
- **fold 평균 AUROC:** fold마다 모델이 달라 점수 크기가 다르다. 그래서 fold 안에서 AUROC를 구해 평균한다. 다른 모델의 OOF AUROC(전체를 한 번에 계산)와 계산 방식이 달라서 직접 순위를 매기지 않는다.
- **재현율·오탐률:** STFPM 자체 기준(학습 정상 95% 지점)으로 판정한 결과. 같은 사진 오탐·미탐 수는 다른 모델과 나란히 볼 수 있다.
- 묶음: `common` 3,039 (세 선별 모두 통과) / `common_clean` 2,648 / `common_bed_visible` 391 / `own_stage2` 3,062 / `some_pass` 422 / `none_pass` 539 / `all4000`

**이 셀 출력을 복사해서 Claude에게 붙여 주면 된다.**

In [ ]:
import pandas as pd, numpy as np
from sklearn.metrics import roc_auc_score
sel = pd.read_csv("/content/grid/selections.csv", encoding="utf-8-sig").set_index("source")
tex = pd.read_csv("/content/texpatch/texture/texture_manifest.csv", encoding="utf-8-sig").set_index("source")
sel["texture_gate"] = tex.texture_gate_accepted.reindex(sel.index)
sel["common"] = (sel.stage2 == 1) & (sel.recycle == 1) & (sel.texture_gate == 1)
anyp = (sel.stage2 == 1) | (sel.recycle == 1) | (sel.texture_gate == 1)
assert sel.common.sum() == 3039
GROUPS = {"common": sel.common, "common_clean": sel.common & (sel.clean_old == 1),
          "common_bed_visible": sel.common & (sel.clean_old == 0), "own_stage2": sel.stage2 == 1,
          "some_pass": anyp & ~sel.common, "none_pass": ~anyp, "all4000": sel.common | True}
def load(cond):
    d = pd.concat([pd.read_csv(f"{RESULTS}/{cond}/fold{k}/scores.csv", encoding="utf-8-sig") for k in range(5)
                   if os.path.exists(f"{RESULTS}/{cond}/fold{k}/scores.csv")])
    d["source"] = d.source.str.replace(chr(92), "/"); return d
rows = []
for cond in ("s2_raw_mean", "s2_texture_topk"):
    if not os.path.exists(f"{RESULTS}/{cond}"): continue
    d = load(cond)
    for name, mask in GROUPS.items():
        g = d[d.source.map(mask)]
        v = [roc_auc_score(x.label_id, x.score) for _, x in g.groupby("fold") if x.label_id.nunique() == 2]
        y, p = g.label_id, g.pred
        rows.append({"조건": cond, "묶음": name, "n": len(g), "fold 수": d.fold.nunique(),
                     "fold평균 AUROC": round(np.mean(v), 4) if v else None, "fold별 AUROC": " ".join(f"{x:.3f}" for x in v),
                     "오탐(FP)": int(((p == 1) & (y == 0)).sum()), "미탐(FN)": int(((p == 0) & (y == 1)).sum()),
                     "언더필 재현율": round(((p == 1) & (y == 1)).sum() / max((y == 1).sum(), 1), 4),
                     "정상 오탐률": round(((p == 1) & (y == 0)).sum() / max((y == 0).sum(), 1), 4)})
table = pd.DataFrame(rows); table.to_csv(f"{RESULTS}/summary_groups.csv", index=False, encoding="utf-8-sig")
print(table.to_string(index=False))

## 9. 결과 묶기
점수·로그·설정만 Drive `results/stfpm_stage2v32_20261013_scores.zip`으로 묶는다(체크포인트 제외). 이 zip을 Claude에게 주면 된다.

In [ ]:
import zipfile, glob
ZIP = f"{DRIVE_DIR}/results/stfpm_stage2v32_20261013_scores.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{RESULTS}/**/*", recursive=True):
        if p.endswith((".csv", ".json", ".txt", ".yaml")):
            z.write(p, os.path.relpath(p, RESULTS))
print(ZIP, f"{os.path.getsize(ZIP) / 2**20:.1f} MiB")

## 10. 런타임 반납

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()